# SVM Prediction with SHAP Analysis (No Binning)

This notebook performs prediction using an SVM with RBF kernel and RandomizedSearchCV. 
It uses standard preprocessing (Imputation + Scaling) instead of binning, and includes SHAP analysis.

## 1. Environment Setup

In [1]:
%pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
import shap

# Ensure plots are displayed inline
%matplotlib inline

## 2. Data Loading

In [3]:
# Setup Kaggle API key if not already present
if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
    %mkdir -p ~/.kaggle
    %cp kaggle.json ~/.kaggle/
    %chmod 600 ~/.kaggle/kaggle.json

# Download dataset if not present
if not os.path.exists('data/heloc_dataset_v1.csv'):
    !mkdir -p data
    !kaggle datasets download -d averkiyoliabev/home-equity-line-of-creditheloc
    !unzip -o home-equity-line-of-creditheloc.zip -d data/
    # Rename if necessary
    if os.path.exists('data/heloc_dataset_v1 (1).csv'):
        !mv "data/heloc_dataset_v1 (1).csv" "data/heloc_dataset_v1.csv"

The syntax of the command is incorrect.
UsageError: Line magic function `%cp` not found.


In [4]:
df = pd.read_csv("data/heloc_dataset_v1.csv")
df.head()

,RiskPerformance,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,...,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,Bad,55,144,4,84,20,3,0,83,2,...,43,0,0,0,33,-8,8,1,1,69
1,Bad,61,58,15,41,2,4,4,100,-7,...,67,0,0,0,0,-8,0,-8,-8,0
2,Bad,67,66,5,24,9,0,0,100,-7,...,44,0,4,4,53,66,4,2,1,86
3,Bad,66,169,1,73,28,1,1,93,76,...,57,0,5,4,72,83,6,4,3,91
4,Bad,81,333,27,132,12,0,0,100,-7,...,25,0,1,1,51,89,3,1,0,80


## 3. Preprocessing

In [5]:
# Define variable names (exclude target)
variable_names = list(df.columns[1:])
X = df[variable_names].values

# Encode target
y = df.RiskPerformance.values
mask = y == "Bad"
y[mask] = 1
y[~mask] = 0
y = y.astype(int)

print(f"Target distribution:\n{pd.Series(y).value_counts(normalize=True)}")

Target distribution:
1    0.521943
0    0.478057
Name: proportion, dtype: float64


In [6]:
# Handle special codes (-9, -8, -7) as missing values
special_codes = [-9, -8, -7]
X = np.where(np.isin(X, special_codes), np.nan, X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 4. Model Training (SVM with RBF Kernel)

In [7]:
# Pipeline: Imputation -> Scaling -> SVM
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True))
])

# Expanded Hyperparameter tuning with RandomizedSearchCV
param_dist = {
    'svm__C': [0.1, 1, 10, 100, 1000],
    'svm__gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
    'svm__class_weight': [None, 'balanced']
}

random_search = RandomizedSearchCV(
    pipeline, 
    param_distributions=param_dist, 
    n_iter=15, # Increased iterations for better exploration
    cv=3, 
    scoring='roc_auc', 
    n_jobs=-1, 
    random_state=42,
    verbose=2
)

print("Starting RandomizedSearchCV with expanded parameters...")
random_search.fit(X_train, y_train)

print("Best Parameters:", random_search.best_params_)
print("Best ROC AUC:", random_search.best_score_)

Starting RandomizedSearchCV with expanded parameters...
Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best Parameters: {'svm__gamma': 0.01, 'svm__class_weight': None, 'svm__C': 0.1}
Best ROC AUC: 0.7972283367787828


In [ ]:
pipeline = Pipeline([
    ('imputer', KNNImputer()),
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True))
])

# Expanded Hyperparameter grid for GridSearchCV
param_grid = {
    # KNN Imputer parameters
    'imputer__n_neighbors': [3],
    'imputer__weights': ['distance'],

    # SVM parameters
    'svm__C': [0.1],
    'svm__gamma': [0.01],
    'svm__class_weight': [None],
    'svm__degree': [3],  # Only used for poly kernel
    'svm__coef0': [0.0]  # Used for poly and sigmoid kernels
}

grid_search = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

print("Starting GridSearchCV with expanded parameters...")
grid_search.fit(X_train, y_train)

print("Best Parameters:", grid_search.best_params_)
print("Best ROC AUC:", grid_search.best_score_)

Starting GridSearchCV with expanded parameters...
Fitting 3 folds for each of 216 candidates, totalling 648 fits
Best Parameters: {'imputer__n_neighbors': 3, 'imputer__weights': 'distance', 'svm__C': 0.1, 'svm__class_weight': None, 'svm__coef0': 0.0, 'svm__degree': 2, 'svm__gamma': 0.01}
Best ROC AUC: 0.7288155850364527


In [12]:
# Evaluate on Test Set
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred))
print(f"Test ROC AUC: {roc_auc_score(y_test, y_prob):.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.70      0.65      0.67      1004
           1       0.70      0.75      0.72      1088

    accuracy                           0.70      2092
   macro avg       0.70      0.70      0.70      2092
weighted avg       0.70      0.70      0.70      2092

Test ROC AUC: 0.7701


## 5. SHAP Analysis

In [13]:
# Preprocess training data for SHAP background (Impute + Scale)
# We need to manually apply the steps before the model to pass to KernelExplainer
preprocessor = Pipeline([
    ('imputer', best_model.named_steps['imputer']),
    ('scaler', best_model.named_steps['scaler'])
])

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# Summarize background data
X_train_summary = shap.kmeans(X_train_processed, 10)

# Use the SVM part of the pipeline for explanation
svm_model = best_model.named_steps['svm']
explainer = shap.KernelExplainer(svm_model.predict_proba, X_train_summary)

# Calculate SHAP values for a subset of the test set
X_test_subset = X_test_processed[:50]
shap_values = explainer.shap_values(X_test_subset)

shap_values_class1 = shap_values[1]

  0%|          | 0/50 [00:00<?, ?it/s]

In [14]:
# SHAP Summary Plot
plt.figure()
shap.summary_plot(shap_values_class1, X_test_subset, feature_names=variable_names)
plt.show()

AssertionError: The shape of the shap_values matrix does not match the shape of the provided data matrix.

<Figure size 640x480 with 0 Axes>

In [ ]:
# SHAP Dependence Plot for the top feature
mean_abs_shap = np.mean(np.abs(shap_values_class1), axis=0)
top_feature_idx = np.argmax(mean_abs_shap)
top_feature_name = variable_names[top_feature_idx]

print(f"Top feature: {top_feature_name}")

shap.dependence_plot(top_feature_name, shap_values_class1, X_test_subset, feature_names=variable_names)

In [ ]:
# SHAP Force Plot for a single prediction
shap.initjs()
idx = 0
shap.force_plot(explainer.expected_value[1], shap_values_class1[idx], X_test_subset[idx], feature_names=variable_names)